In [5]:
import cv2
import numpy as np
import os

# Generator function for loading images and binary masks
def image_mask_generator(image_dir, label_dir, batch_size, img_shape=(128, 128)):
    image_files = os.listdir(image_dir)
    num_samples = len(image_files)
    
    while True:  # Infinite loop to continuously yield batches
        np.random.shuffle(image_files)  # Shuffle the image file order at the start of each epoch

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_images = np.zeros((end_idx - start_idx, *img_shape, 3), dtype=np.float32)
            batch_masks = np.zeros((end_idx - start_idx, *img_shape), dtype=np.float32)
            
            # Process each image and its corresponding label in the batch
            for i, file_name in enumerate(image_files[start_idx:end_idx]):
                # Load the image from the image directory
                img_path = os.path.join(image_dir, file_name)
                image = cv2.imread(img_path)
                image = cv2.resize(image, img_shape)  # Resize image to input shape
                image = image / 255.0  # Normalize image to [0, 1]
                
                # Initialize mask for the current image
                mask = np.zeros(img_shape, dtype=np.float32)
                
                # Load corresponding label file and process the bounding boxes
                label_file = os.path.join(label_dir, os.path.splitext(file_name)[0] + '.txt')
                with open(label_file, 'r') as f:
                    for line in f:
                        # Parse each line as: class_id, x_center, y_center, width, height (relative coordinates)
                        _, x_center, y_center, width, height = map(float, line.strip().split())
                        
                        # Convert relative coordinates to absolute pixel values
                        x_min = int((x_center - width / 2) * img_shape[1])  # Image width
                        y_min = int((y_center - height / 2) * img_shape[0])  # Image height
                        x_max = int((x_center + width / 2) * img_shape[1])
                        y_max = int((y_center + height / 2) * img_shape[0])

                        # Draw the bounding box on the mask
                        mask[y_min:y_max, x_min:x_max] = 1  # Set the pixels inside the bounding box to 1 (foreground)

                # Store the processed image and mask in the batch
                batch_images[i] = image
                batch_masks[i] = mask  # Mask is 2D for binary classification

            # Expand mask dimensions to match the output format (for binary classification)
            batch_masks = np.expand_dims(batch_masks, axis=-1)  # Add a channel dimension to masks
            
            yield batch_images, batch_masks  # Yield the batch of images and masks

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Residual Block
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, shortcut])  # Add the shortcut connection
    x = layers.ReLU()(x)
    return x

# Build ResNet Model for Segmentation
def build_resnet(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    # Initial Convolution and Max Pooling
    x = layers.Conv2D(64, kernel_size=(7, 7), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2, padding='same')(x)

    # Residual Blocks
    for _ in range(3):
        x = residual_block(x, 64)

    # Upsampling Layer
    x = layers.Conv2DTranspose(64, kernel_size=(2, 2), strides=(2, 2), padding='same')(x)
    
    # Final Convolution to get the output shape
    outputs = layers.Conv2D(1, kernel_size=(1, 1), activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)  # Output shape will be (None, 128, 128, 1)
    return model

# Compile the model
input_shape = (128, 128, 3)  # Adjust based on your input data
resnet_model = build_resnet(input_shape)
resnet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [14]:
import os
import tensorflow as tf
from tensorflow.keras import callbacks

# Directories for images and labels
image_dir = 'D:\\Shivani\\Minors 3\\datasets\\train\\images'  # Folder containing the image files
label_dir = 'D:\\Shivani\\Minors 3\\datasets\\train\\labels'  # Folder containing the label files

batch_size = 32
img_shape = (128, 128)  # Example image size

# Create the data generator
train_gen = image_mask_generator(image_dir, label_dir, batch_size, img_shape)

# Build and compile the ResNet model
input_shape = (*img_shape, 3)  # Image size with 3 channels (128, 128, 3)
resnet_model = build_resnet(input_shape)

# Compile the model
resnet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks for training
early_stopping = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(factor=0.2, patience=3)

# Train the ResNet model using the generator
resnet_model.fit(train_gen,
                 steps_per_epoch=len(os.listdir(image_dir)) // batch_size,
                 epochs=50,
                 callbacks=[early_stopping, reduce_lr])


Epoch 1/50
 13/269 ━━━━━━━━━━━━━━━━━━━━ 6:00 1s/step - accuracy: 0.7460 - loss: 0.5054

KeyboardInterrupt: 